# FIFA World Cup 2026 — Analytic Task

**Research question.** Among outfield players who played at least 90 minutes, do young players commit more fouls per 90 minutes than veteran players?

- $H_0$: $\mu_{\text{Young}} \leq \mu_{\text{Veteran}}$
- $H_1$: $\mu_{\text{Young}} > \mu_{\text{Veteran}}$

The direction was specified in advance, so the test is one-tailed at $\alpha = 0.05$.

**Input file:** `wc2026_discipline.csv`


### Importing all the necessary libraries


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

### Reading the dataset file and setting the output folder


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    FILENAME = next(iter(uploaded))
except ImportError:
    FILENAME = 'wc2026_discipline.csv'

FIGDIR = 'figures'
os.makedirs(FIGDIR, exist_ok=True)

### Defining the plotting style and the colour palette


In [ ]:
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
})

C_YOUNG, C_PEAK, C_VET = '#4C72B0', '#B0B0B0', '#C44E52'
PAL = {'Young': C_YOUNG, 'Peak': C_PEAK, 'Veteran': C_VET}
ORDER = ['Young', 'Peak', 'Veteran']
GROUPS = ['Young', 'Veteran']

### Helper function for saving each figure


In [ ]:
def save(fig, name):
    fig.savefig(f'{FIGDIR}/{name}.png', bbox_inches='tight')
    plt.close(fig)
    print('  saved', name)

### Helper function for the confidence interval of a mean


In [ ]:
def ci(s, conf=0.95):
    return stats.t.interval(conf, len(s) - 1, s.mean(), stats.sem(s))

### Helper function for Welch's two-sample t-test


In [ ]:
def welch(a, b):
    t, p2 = stats.ttest_ind(a, b, equal_var=False)
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    se = np.sqrt(va / na + vb / nb)
    dof = se ** 4 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    diff = a.mean() - b.mean()
    lo, hi = stats.t.interval(0.95, dof, diff, se)
    p1 = p2 / 2 if t > 0 else 1 - p2 / 2
    return dict(t=t, p2=p2, p1=p1, dof=dof, diff=diff, lo=lo, hi=hi,
                na=na, nb=nb)

### Helper function for printing section headings


In [ ]:
def rule(txt):
    print('\n' + '=' * 68)
    print(txt)
    print('=' * 68)

### Loading the dataset and deriving the working columns


In [ ]:
df = pd.read_csv(FILENAME)
df['age_group'] = pd.Categorical(df['age_group'], categories=ORDER,
                                 ordered=True)
df['pos_simple'] = df['Pos'].str[:2]
df['is_hybrid'] = df['Pos'].str.len() > 2
df['code'] = df['Squad'].str.split(' ').str[0]
df['squad_name'] = df['Squad'].str.split(' ', n=1).str[1]

### Mapping each national squad to its confederation


In [ ]:
CONF = {
    'ar': 'CONMEBOL', 'br': 'CONMEBOL', 'co': 'CONMEBOL', 'ec': 'CONMEBOL',
    'py': 'CONMEBOL', 'uy': 'CONMEBOL',
    'at': 'UEFA', 'ba': 'UEFA', 'be': 'UEFA', 'ch': 'UEFA', 'cz': 'UEFA',
    'de': 'UEFA', 'eng': 'UEFA', 'es': 'UEFA', 'fr': 'UEFA', 'hr': 'UEFA',
    'nl': 'UEFA', 'no': 'UEFA', 'pt': 'UEFA', 'sct': 'UEFA', 'se': 'UEFA',
    'tr': 'UEFA',
    'cd': 'CAF', 'ci': 'CAF', 'cv': 'CAF', 'dz': 'CAF', 'eg': 'CAF',
    'gh': 'CAF', 'ma': 'CAF', 'sn': 'CAF', 'tn': 'CAF', 'za': 'CAF',
    'iq': 'AFC', 'ir': 'AFC', 'jo': 'AFC', 'jp': 'AFC', 'kr': 'AFC',
    'qa': 'AFC', 'sa': 'AFC', 'uz': 'AFC', 'au': 'AFC',
    'ca': 'CONCACAF', 'cw': 'CONCACAF', 'ht': 'CONCACAF', 'mx': 'CONCACAF',
    'pa': 'CONCACAF', 'us': 'CONCACAF',
    'nz': 'OFC',
}
df['confederation'] = df['code'].map(CONF)

### Inspecting the dataset and summarising it by age group


In [ ]:
rule('1. DATASET')
print(f'Rows: {len(df)}   Squads: {df["squad_name"].nunique()}   '
      f'Missing values: {int(df.isna().sum().sum())}')
print(f'Minutes: {df["Min"].min()} to {df["Min"].max()}')
print(f'Hybrid position labels: {int(df["is_hybrid"].sum())}')
print('\nBy age group:')
print(df.groupby('age_group', observed=True)['fouls_per_90']
      .agg(n='count', mean='mean', sd='std', median='median',
           q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75),
           min='min', max='max').round(3).to_string())

### Drawing the stratified random sample


In [ ]:
pair = df[df['age_group'].isin(GROUPS)].copy()
pair['age_group'] = pair['age_group'].cat.remove_unused_categories()

N = int(pair['age_group'].value_counts().min())
sample = pd.concat([g.sample(n=N, random_state=42)
                    for _, g in pair.groupby('age_group', observed=True)])

y = sample.loc[sample['age_group'] == 'Young', 'fouls_per_90']
v = sample.loc[sample['age_group'] == 'Veteran', 'fouls_per_90']
Y_POP = pair.loc[pair['age_group'] == 'Young', 'fouls_per_90']
V_POP = pair.loc[pair['age_group'] == 'Veteran', 'fouls_per_90']

rule('2. SAMPLING')
print(f'Population (Young + Veteran): {len(pair)}')
print(f'Stratified sample: {len(sample)}  ({N} per group, seed 42)')

### Descriptive statistics for the sample


In [ ]:
rule('3. DESCRIPTIVE STATISTICS (sample)')
print(sample.groupby('age_group', observed=True)['fouls_per_90']
      .agg(n='count', mean='mean', sd='std', median='median',
           q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75),
           skew='skew').round(3).to_string())

### Confidence intervals for the group means and the difference


In [ ]:
res = welch(y, v)
res_pop = welch(Y_POP, V_POP)

rule('4. CONFIDENCE INTERVALS (95%)')
for g, s in [('Young', y), ('Veteran', v)]:
    lo, hi = ci(s)
    print(f'  {g:<9} mean {s.mean():.3f}   CI [{lo:.3f}, {hi:.3f}]')
print(f"\n  Difference (Young - Veteran) = {res['diff']:+.3f}")
print(f"  95% CI [{res['lo']:.3f}, {res['hi']:.3f}]")

### Two-sample t-test and the assumption checks


In [ ]:
lev = stats.levene(y, v)
sh_y, sh_v = stats.shapiro(y).pvalue, stats.shapiro(v).pvalue
mw = stats.mannwhitneyu(y, v, alternative='greater')

rule('5. TWO-SAMPLE t-TEST (Welch, one-tailed)')
print(f"  t = {res['t']:.3f}   df = {res['dof']:.1f}")
print(f"  one-tailed p = {res['p1']:.4f}   (two-tailed {res['p2']:.4f})")
print(f"  Decision at alpha = 0.05: "
      f"{'reject H0' if res['p1'] < 0.05 else 'fail to reject H0'}")
print(f"\n  Full population instead of sample: t = {res_pop['t']:.3f}, "
      f"one-tailed p = {res_pop['p1']:.4f}")
print('\n  Assumption checks')
print(f"    Levene equal variance   p = {lev.pvalue:.3f}")
print(f"    Shapiro normality       Young p = {sh_y:.2e}, "
      f"Veteran p = {sh_v:.2e}")
print(f"    Mann-Whitney U (robust) p = {mw.pvalue:.4f}")

### Robustness check: sensitivity to the minutes threshold


In [ ]:
rule('6. ROBUSTNESS CHECKS')
print('  Sensitivity to the minutes threshold')
print(f"  {'min':>5} {'nY':>5} {'nV':>5} {'Young':>8} {'Veteran':>8} "
      f"{'diff':>8} {'p(1)':>8}")
thr_rows = []
for thr in [90, 135, 180, 225, 270, 315, 360]:
    q = pair[pair['Min'] >= thr]
    a = q.loc[q['age_group'] == 'Young', 'fouls_per_90']
    b = q.loc[q['age_group'] == 'Veteran', 'fouls_per_90']
    if len(a) > 3 and len(b) > 3:
        r = welch(a, b)
        thr_rows.append((thr, r))
        print(f"  {thr:>5} {len(a):>5} {len(b):>5} {a.mean():>8.3f} "
              f"{b.mean():>8.3f} {r['diff']:>+8.3f} {r['p1']:>8.4f}")

### Robustness check: the effect within each position


In [ ]:
print('\n  Within position (hybrid labels excluded)')
pos_rows = []
for p in ['DF', 'MF', 'FW']:
    q = pair[(pair['pos_simple'] == p) & (~pair['is_hybrid'])]
    a = q.loc[q['age_group'] == 'Young', 'fouls_per_90']
    b = q.loc[q['age_group'] == 'Veteran', 'fouls_per_90']
    if len(a) > 3 and len(b) > 3:
        r = welch(a, b)
        pos_rows.append((p, r))
        print(f'  {p}  nY={len(a):<4} nV={len(b):<4} '
              f"Young {a.mean():.3f}  Veteran {b.mean():.3f}  "
              f"diff {r['diff']:+.3f}  p(1) {r['p1']:.4f}")

### Robustness check: the effect within each confederation


In [ ]:
print('\n  By confederation (40+ players in the Young/Veteran pool)')
confs = pair['confederation'].value_counts().loc[lambda s: s >= 40].index.tolist()
conf_rows = []
for c in confs:
    q = pair[pair['confederation'] == c]
    a = q.loc[q['age_group'] == 'Young', 'fouls_per_90']
    b = q.loc[q['age_group'] == 'Veteran', 'fouls_per_90']
    if len(a) > 3 and len(b) > 3:
        r = welch(a, b)
        conf_rows.append((c, r))
        print(f'  {c:<10} nY={len(a):<4} nV={len(b):<4} '
              f"diff {r['diff']:+.3f}  p(1) {r['p1']:.4f}")

### Rendering the figures


In [ ]:
rule('FIGURES')

### Figure 1: Composition of the analysis population


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
ac = df['age_group'].value_counts().reindex(ORDER)
axes[0].bar(ac.index, ac.values, color=[PAL[i] for i in ac.index],
            edgecolor='white')
for i, val in enumerate(ac.values):
    axes[0].text(i, val + 6, str(val), ha='center', fontsize=9)
axes[0].set_ylabel('Players')
axes[0].set_title('By age group')

pc = df['pos_simple'].value_counts()
axes[1].bar(pc.index, pc.values, color='#6C8EBF', edgecolor='white')
axes[1].set_title('By position (hybrids folded in)')

cc = df['confederation'].value_counts()
axes[2].barh(cc.index[::-1], cc.values[::-1], color='#6C8EBF',
             edgecolor='white')
axes[2].grid(axis='y', visible=False)
axes[2].set_title('By confederation')
fig.suptitle('Figure 1: Composition of the analysis population '
             f'(n = {len(df)})', y=1.04)
save(fig, 'fig01_composition')

### Figure 2: Minutes played and the 90-minute cutoff


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(df['Min'], bins=35, color='#B0B0B0', edgecolor='white')
axes[0].axvline(90, color=C_VET, lw=2, ls='--')
axes[0].text(100, axes[0].get_ylim()[1] * 0.88, '90-minute cutoff',
             color=C_VET, fontsize=9)
axes[0].set_xlabel('Total minutes played')
axes[0].set_ylabel('Players')
axes[0].set_title('Minutes played')

axes[1].scatter(df['Min'], df['fouls_per_90'], s=14, alpha=0.35,
                color='#6C8EBF', edgecolor='none')
axes[1].set_xlabel('Total minutes played')
axes[1].set_ylabel('Fouls per 90 minutes')
axes[1].set_title('Low-minute players produce extreme rates')
fig.suptitle('Figure 2: Why the minutes filter matters', y=1.03)
save(fig, 'fig02_minutes')

### Figure 3: Age distribution and the group boundaries


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for g in ORDER:
    ax.hist(df.loc[df['age_group'] == g, 'Age'], bins=range(17, 42),
            color=PAL[g], alpha=0.9, label=g, edgecolor='white')
ax.axvline(23.5, color='k', lw=1, ls=':')
ax.axvline(29.5, color='k', lw=1, ls=':')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Players')
ax.set_title('Figure 3: Age distribution and the group boundaries')
ax.legend()
save(fig, 'fig03_age_bands')

### Figure 4: Comparing the sample against the population


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(pair['fouls_per_90'], bins=25, density=True, color='#B0B0B0',
             edgecolor='white', label=f'Population (n={len(pair)})')
axes[0].hist(sample['fouls_per_90'], bins=25, density=True, histtype='step',
             lw=2, color=C_VET, label=f'Sample (n={len(sample)})')
axes[0].set_xlabel('Fouls per 90 minutes')
axes[0].set_ylabel('Density')
axes[0].set_title('Response variable')
axes[0].legend(fontsize=8)

axes[1].hist(pair['Age'], bins=range(17, 42), density=True, color='#B0B0B0',
             edgecolor='white', label='Population')
axes[1].hist(sample['Age'], bins=range(17, 42), density=True, histtype='step',
             lw=2, color=C_VET, label='Sample')
axes[1].set_xlabel('Age (years)')
axes[1].set_title('Age')
axes[1].legend(fontsize=8)
fig.suptitle('Figure 4: The stratified sample tracks the population', y=1.03)
save(fig, 'fig04_sample_vs_pop')

### Figure 5: Distribution of fouls per 90 minutes


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(df['fouls_per_90'], bins=40, color='#6C8EBF', edgecolor='white')
ax.axvline(df['fouls_per_90'].mean(), color=C_VET, lw=2,
           label=f"Mean = {df['fouls_per_90'].mean():.2f}")
ax.axvline(df['fouls_per_90'].median(), color='k', lw=2, ls='--',
           label=f"Median = {df['fouls_per_90'].median():.2f}")
ax.set_xlabel('Fouls per 90 minutes')
ax.set_ylabel('Players')
ax.set_title('Figure 5: Fouls per 90 is right-skewed with a floor at zero '
             f"(skew = {df['fouls_per_90'].skew():.2f})")
ax.legend(fontsize=8)
save(fig, 'fig05_response_hist')

### Figure 6: Boxplot, violin and individual players


In [ ]:
data = [sample.loc[sample['age_group'] == g, 'fouls_per_90'] for g in GROUPS]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

bp = axes[0].boxplot(data, tick_labels=GROUPS, patch_artist=True, widths=.55)
for patch, g in zip(bp['boxes'], GROUPS):
    patch.set_facecolor(PAL[g]); patch.set_alpha(.75)
for med in bp['medians']:
    med.set_color('k'); med.set_linewidth(1.6)
axes[0].set_ylabel('Fouls per 90 minutes')
axes[0].set_title('Boxplot')

vp = axes[1].violinplot(data, showmedians=True, widths=.7)
for body, g in zip(vp['bodies'], GROUPS):
    body.set_facecolor(PAL[g]); body.set_alpha(.7)
axes[1].set_xticks([1, 2]); axes[1].set_xticklabels(GROUPS)
axes[1].set_title('Violin')

rng = np.random.default_rng(1)
for i, (g, d) in enumerate(zip(GROUPS, data), start=1):
    axes[2].scatter(i + rng.normal(0, .06, len(d)), d, s=16, alpha=.5,
                    color=PAL[g], edgecolor='none')
    axes[2].hlines(d.mean(), i - .25, i + .25, color='k', lw=2)
axes[2].set_xticks([1, 2]); axes[2].set_xticklabels(GROUPS)
axes[2].set_title('Individual players (mean marked)')
fig.suptitle('Figure 6: Fouls per 90 by age group — three views', y=1.04)
save(fig, 'fig06_three_views')

### Figure 7: Kernel density and empirical CDF


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
xs = np.linspace(0, sample['fouls_per_90'].max() * 1.05, 300)
for g, d in zip(GROUPS, data):
    kde = stats.gaussian_kde(d)
    axes[0].plot(xs, kde(xs), color=PAL[g], lw=2, label=g)
    axes[0].fill_between(xs, kde(xs), color=PAL[g], alpha=.18)
    axes[1].step(np.sort(d), np.arange(1, len(d) + 1) / len(d),
                 color=PAL[g], lw=2, label=g)
axes[0].set_xlabel('Fouls per 90 minutes'); axes[0].set_ylabel('Density')
axes[0].set_title('Kernel density'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Fouls per 90 minutes')
axes[1].set_ylabel('Cumulative proportion')
axes[1].set_title('Empirical CDF'); axes[1].legend(fontsize=8)
fig.suptitle('Figure 7: Comparing the full shape of both distributions',
             y=1.03)
save(fig, 'fig07_density_ecdf')

### Figure 8: Mean fouls per 90 across all three age bands


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
d3 = [df.loc[df['age_group'] == g, 'fouls_per_90'] for g in ORDER]
for i, (g, d) in enumerate(zip(ORDER, d3)):
    lo, hi = ci(d)
    ax.errorbar(i, d.mean(), yerr=[[d.mean() - lo], [hi - d.mean()]],
                fmt='o', ms=10, capsize=7, color=PAL[g], lw=2)
    ax.text(i + .1, d.mean(), f'{d.mean():.3f}\nn={len(d)}', fontsize=8,
            va='center')
ax.plot(range(3), [d.mean() for d in d3], color='k', lw=1, ls='--', alpha=.5)
ax.set_xticks(range(3)); ax.set_xticklabels(ORDER)
ax.set_xlim(-.4, 2.5)
ax.set_ylabel('Mean fouls per 90 minutes')
ax.set_title('Figure 8: The gradient runs across all three age bands\n'
             '(95% CI; Peak is excluded from the t-test)')
save(fig, 'fig08_three_bands')

### Figure 9: Age treated as a continuous variable


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.scatter(df['Age'], df['fouls_per_90'], s=df['Min'] / 12, alpha=.35,
           color='#6C8EBF', edgecolor='none')
m, b = np.polyfit(df['Age'], df['fouls_per_90'], 1)
xg = np.linspace(df['Age'].min(), df['Age'].max(), 50)
ax.plot(xg, m * xg + b, color=C_VET, lw=2,
        label=f'OLS fit: {m:+.4f} fouls/90 per year of age')
binned = df.groupby('Age')['fouls_per_90'].mean()
ax.plot(binned.index, binned.values, 'k.-', lw=1, ms=5, alpha=.65,
        label='Mean by exact age')
r, pr = stats.pearsonr(df['Age'], df['fouls_per_90'])
ax.set_xlabel('Age (years)'); ax.set_ylabel('Fouls per 90 minutes')
ax.set_title('Figure 9: Age against fouling, treated continuously\n'
             f'(r = {r:.3f}, p = {pr:.3f}; point size = minutes played)')
ax.legend(fontsize=8)
save(fig, 'fig09_age_continuous')

### Figure 10: Confidence intervals for the means and the difference


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for i, (g, d) in enumerate(zip(GROUPS, data)):
    lo, hi = ci(d)
    axes[0].errorbar(i, d.mean(), yerr=[[d.mean() - lo], [hi - d.mean()]],
                     fmt='o', ms=10, capsize=7, color=PAL[g], lw=2)
    axes[0].text(i + .09, d.mean(), f'{d.mean():.3f}\n[{lo:.3f}, {hi:.3f}]',
                 fontsize=8, va='center')
axes[0].set_xticks(range(2)); axes[0].set_xticklabels(GROUPS)
axes[0].set_xlim(-.5, 1.7)
axes[0].set_ylabel('Mean fouls per 90 minutes')
axes[0].set_title('Group means, 95% CI')

axes[1].errorbar(res['diff'], 0,
                 xerr=[[res['diff'] - res['lo']], [res['hi'] - res['diff']]],
                 fmt='o', ms=11, capsize=7, color='k', lw=2)
axes[1].axvline(0, color=C_VET, lw=2, ls='--')
axes[1].text(0, .3, 'no difference', color=C_VET, ha='center', fontsize=9)
axes[1].set_yticks([]); axes[1].set_ylim(-.5, .5)
axes[1].set_xlabel('Young mean − Veteran mean (fouls per 90)')
axes[1].set_title(f"Difference = {res['diff']:+.3f}\n"
                  f"95% CI [{res['lo']:.3f}, {res['hi']:.3f}]")
fig.suptitle('Figure 10: Confidence intervals', y=1.04)
save(fig, 'fig10_confidence_intervals')

### Figure 11: Bootstrap check on the confidence interval


In [ ]:
rng = np.random.default_rng(42)
yv, vv = y.values, v.values
boot = np.array([rng.choice(yv, len(yv), True).mean()
                 - rng.choice(vv, len(vv), True).mean()
                 for _ in range(5000)])
blo, bhi = np.percentile(boot, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(boot, bins=50, color='#B0B0B0', edgecolor='white')
ax.axvline(0, color=C_VET, lw=2, ls='--')
ax.axvline(blo, color='k', lw=1.5); ax.axvline(bhi, color='k', lw=1.5)
ax.set_xlabel('Bootstrapped difference in means')
ax.set_ylabel('Frequency')
ax.set_title('Figure 11: Bootstrap check — '
             f'95% percentile interval [{blo:.3f}, {bhi:.3f}]\n'
             f'{(boot <= 0).mean() * 100:.1f}% of resamples fall at or '
             'below zero (5,000 resamples)')
save(fig, 'fig11_bootstrap')

### Figure 12: The one-tailed Welch t-test


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
xs = np.linspace(-4.5, 4.5, 600)
ax.plot(xs, stats.t.pdf(xs, res['dof']), color='k', lw=1.6)
crit = stats.t.ppf(.95, res['dof'])
ax.fill_between(xs, stats.t.pdf(xs, res['dof']), where=(xs >= crit),
                color=C_VET, alpha=.35,
                label=f'Rejection region, t > {crit:.2f} (α = 0.05)')
ax.axvline(res['t'], color=C_YOUNG, lw=2.5,
           label=f"Observed t = {res['t']:.3f}")
ax.set_xlabel(f"t statistic (df = {res['dof']:.1f})")
ax.set_ylabel('Density')
ax.set_title('Figure 12: One-tailed Welch t-test — '
             f"p = {res['p1']:.4f}")
ax.legend(fontsize=8)
save(fig, 'fig12_ttest')

### Figure 13: Q-Q plots and the equal variance check


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for i, (g, d) in enumerate(zip(GROUPS, data)):
    stats.probplot(d, dist='norm', plot=axes[i])
    axes[i].get_lines()[0].set(color=PAL[g], ms=4)
    axes[i].get_lines()[1].set(color='k', lw=1.5)
    axes[i].set_title(f'Q–Q plot: {g}  (Shapiro p = '
                      f'{stats.shapiro(d).pvalue:.1e})')
axes[2].bar(GROUPS, [d.std(ddof=1) for d in data],
            color=[PAL[g] for g in GROUPS], edgecolor='white')
axes[2].set_ylabel('Standard deviation')
axes[2].set_title(f"Levene's test: p = {lev.pvalue:.3f}")
fig.suptitle('Figure 13: Assumption checks — skewed data motivates the '
             'Mann–Whitney cross-check', y=1.04)
save(fig, 'fig13_assumptions')

### Figure 14: The effect within each position


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (p, r) in zip(axes, pos_rows):
    q = pair[(pair['pos_simple'] == p) & (~pair['is_hybrid'])]
    dd = [q.loc[q['age_group'] == g, 'fouls_per_90'] for g in GROUPS]
    bp = ax.boxplot(dd, tick_labels=GROUPS, patch_artist=True, widths=.55)
    for patch, g in zip(bp['boxes'], GROUPS):
        patch.set_facecolor(PAL[g]); patch.set_alpha(.75)
    for med in bp['medians']:
        med.set_color('k')
    ax.set_title(f"{p}  (n = {r['na']}/{r['nb']})\n"
                 f"diff {r['diff']:+.3f}, p = {r['p1']:.3f}")
axes[0].set_ylabel('Fouls per 90 minutes')
fig.suptitle('Figure 14: Direction holds within every position, but no '
             'stratum reaches significance', y=1.06)
save(fig, 'fig14_by_position')

### Figure 15: Position mix of the two age groups


In [ ]:
mix = (pd.crosstab(pair['age_group'], pair['pos_simple'], normalize='index')
       * 100)
fig, ax = plt.subplots(figsize=(7, 3.6))
bottom = np.zeros(len(mix))
for col, colour in zip(['DF', 'MF', 'FW'], ['#4C72B0', '#8FA9C9', '#C9D3E0']):
    if col in mix:
        ax.bar(mix.index.astype(str), mix[col], bottom=bottom, label=col,
               color=colour, edgecolor='white')
        for i, val in enumerate(mix[col]):
            ax.text(i, bottom[i] + val / 2, f'{val:.0f}%', ha='center',
                    fontsize=9)
        bottom += mix[col].values
ax.set_ylabel('Share of group (%)')
ax.set_title('Figure 15: The two age groups have different position mixes\n'
             '— a confounder for the headline result')
ax.legend(fontsize=8)
save(fig, 'fig15_position_mix')

### Figure 16: Forest plot across every subgroup


In [ ]:
strata = [('All players (sample)', res)]
strata += [(f'Position: {p}', r) for p, r in pos_rows]
strata += [(c, r) for c, r in conf_rows]

fig, ax = plt.subplots(figsize=(8, .45 * len(strata) + 2))
ypos = np.arange(len(strata))[::-1]
xmin = min(r['lo'] for _, r in strata)
xmax = max(r['hi'] for _, r in strata)
span = xmax - xmin
ax.set_xlim(min(xmin, 0) - .08 * span, xmax + .22 * span)
for yp, (name, r) in zip(ypos, strata):
    col = 'k' if name.startswith('All') else '#6C8EBF'
    ax.plot([r['lo'], r['hi']], [yp, yp], color=col, lw=2)
    ax.plot(r['diff'], yp, 'o', color=col,
            ms=9 if name.startswith('All') else 6)
    ax.text(r['hi'] + .02 * span, yp, f"n={r['na']}/{r['nb']}", fontsize=8,
            va='center')
ax.axvline(0, color=C_VET, lw=1.6, ls='--')
ax.set_yticks(ypos); ax.set_yticklabels([s[0] for s in strata])
ax.set_xlabel('Young mean − Veteran mean (fouls per 90), 95% CI')
ax.set_title('Figure 16: Forest plot — the effect across every subgroup')
ax.grid(axis='y', visible=False)
save(fig, 'fig16_forest')

### Figure 17: Sensitivity to the minutes threshold


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
thrs = [t for t, _ in thr_rows]
diffs = [r['diff'] for _, r in thr_rows]
los = [r['lo'] for _, r in thr_rows]
his = [r['hi'] for _, r in thr_rows]
ax.errorbar(thrs, diffs,
            yerr=[np.array(diffs) - np.array(los),
                  np.array(his) - np.array(diffs)],
            fmt='o-', ms=7, capsize=5, color='#6C8EBF', lw=2)
ax.axhline(0, color=C_VET, lw=1.8, ls='--')
for t, r in thr_rows:
    ax.text(t, r['hi'] + .02, f"p={r['p1']:.3f}", ha='center', fontsize=7.5)
ax.set_xlabel('Minimum minutes played required for inclusion')
ax.set_ylabel('Young − Veteran (fouls per 90)')
ax.set_title('Figure 17: The effect weakens as the minutes threshold rises\n'
             '— reported as a limitation, not used to select a threshold')
save(fig, 'fig17_threshold_sensitivity')

### Figure 18: Fouls per 90 by age group and confederation


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
w = .36
for j, g in enumerate(GROUPS):
    means, errs = [], []
    for c in confs:
        d = pair[(pair['confederation'] == c) & (pair['age_group'] == g)]['fouls_per_90']
        if len(d) > 2:
            lo, hi = ci(d)
            means.append(d.mean()); errs.append((hi - lo) / 2)
        else:
            means.append(np.nan); errs.append(0)
    ax.bar(np.arange(len(confs)) + (j - .5) * w, means, w, yerr=errs,
           capsize=4, color=PAL[g], edgecolor='white', label=g)
ax.set_xticks(range(len(confs))); ax.set_xticklabels(confs)
ax.set_ylabel('Mean fouls per 90 minutes')
ax.set_title('Figure 18: Fouls per 90 by age group and confederation\n'
             '(40+ players in pool; error bars = 95% CI)')
ax.legend()
save(fig, 'fig18_confederation_bars')

### Figure 19: Heatmap of mean fouls by confederation


In [ ]:
piv = pair.pivot_table(index='confederation', columns='age_group',
                       values='fouls_per_90', aggfunc='mean', observed=True)
cnt = pair.pivot_table(index='confederation', columns='age_group',
                       values='fouls_per_90', aggfunc='count', observed=True)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
im = ax.imshow(piv.values, cmap='RdYlBu_r', aspect='auto')
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        val = piv.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}\n(n={int(cnt.values[i, j])})',
                    ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='Mean fouls per 90')
ax.set_title('Figure 19: Mean fouls per 90 by\nconfederation and age group')
ax.grid(False)
save(fig, 'fig19_heatmap')

### Figure 20: Checking for Simpson's paradox


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for c in confs:
    q = pair[pair['confederation'] == c]
    ms = [q.loc[q['age_group'] == g, 'fouls_per_90'].mean() for g in GROUPS]
    ax.plot([0, 1], ms, 'o-', lw=1.8, ms=7, alpha=.85, label=c)
ax.plot([0, 1], [Y_POP.mean(), V_POP.mean()], 'k--o', lw=3, ms=10,
        label='All players', zorder=5)
ax.set_xticks([0, 1]); ax.set_xticklabels(GROUPS)
ax.set_xlim(-.15, 1.15)
ax.set_ylabel('Mean fouls per 90 minutes')
ax.set_title('Figure 20: Slope by confederation — a line running against\n'
             "the black one would signal Simpson's paradox")
ax.legend(fontsize=8, ncol=2)
save(fig, 'fig20_simpson')

### Figure 21: Squad-level fouling rates


In [ ]:
sq = (df.groupby('squad_name')
      .agg(mean_fouls=('fouls_per_90', 'mean'), n=('Player', 'count'))
      .loc[lambda d: d['n'] >= 8].sort_values('mean_fouls'))
fig, ax = plt.subplots(figsize=(7.5, .22 * len(sq) + 1.5))
ax.barh(sq.index, sq['mean_fouls'], color='#6C8EBF', edgecolor='white')
ax.axvline(df['fouls_per_90'].mean(), color=C_VET, lw=2, ls='--',
           label='Tournament mean')
ax.set_xlabel('Mean fouls per 90 minutes')
ax.set_title('Figure 21: Squad-level fouling rates (8+ eligible players)')
ax.legend(fontsize=8)
ax.grid(axis='y', visible=False)
save(fig, 'fig21_squads')

### Summary table for the write-up


In [ ]:
rule('SUMMARY FOR THE WRITE-UP')
summary = pd.DataFrame({
    'Statistic': ['n', 'Mean fouls/90', 'SD', 'Median',
                  '95% CI lower', '95% CI upper'],
    'Young': [len(y), y.mean(), y.std(ddof=1), y.median(),
              ci(y)[0], ci(y)[1]],
    'Veteran': [len(v), v.mean(), v.std(ddof=1), v.median(),
                ci(v)[0], ci(v)[1]],
}).round(3)
print(summary.to_string(index=False))
print(f"\nDifference {res['diff']:+.3f}, 95% CI "
      f"[{res['lo']:.3f}, {res['hi']:.3f}]")
print(f"Welch t({res['dof']:.1f}) = {res['t']:.3f}, one-tailed p = "
      f"{res['p1']:.4f}")
summary.to_csv('summary_table.csv', index=False)
print(f'\n21 figures written to {FIGDIR}/')

### Archiving the figures folder for download


In [ ]:
import shutil

output_filename = 'figures_archive'
shutil.make_archive(output_filename, 'zip', FIGDIR)

print(f'Successfully created {output_filename}.zip')